

---
### Models Covered in This Notebook
| Phase | Concept | Type |
|-------|---------|------|
| Phase 4A | Decision Tree | Supervised — Classification |
| Phase 4B | Random Forest | Supervised — Ensemble |
| Phase 4C | Logistic Regression | Supervised — Classification |
| Phase 4D | Support Vector Machine (SVM) | Supervised — Classification |
| Phase 4E | Linear Regression (Severity Prediction) | Supervised — Regression |
| Phase 4F | Artificial Neural Network (ANN) | Deep Learning — Keras/TensorFlow |
| Phase 4G | K-Means Clustering + PCA | Unsupervised Learning |
| Phase 4H | CNN (Symptom Grid Representation) | Deep Learning — CNN |
| Phase 4I | Comparative Analysis (All Models) | Model Evaluation |

---
## Phase 1 — Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    mean_squared_error, r2_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

from scipy import stats

import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("All libraries imported successfully!")
print(f"   TensorFlow version : {tf.__version__}")

All libraries imported successfully!
   TensorFlow version : 2.20.0


In [ ]:
from google.colab import files
uploaded = files.upload()
print("\nFiles uploaded:", list(uploaded.keys()))

Saving dataset_updated.csv to dataset_updated (5).csv
Saving symptom_Description_updated.csv to symptom_Description_updated (5).csv
Saving symptom_precaution_updated.csv to symptom_precaution_updated (5).csv
Saving Symptom-severity_updated.csv to Symptom-severity_updated (5).csv

Files uploaded: ['dataset_updated (5).csv', 'symptom_Description_updated (5).csv', 'symptom_precaution_updated (5).csv', 'Symptom-severity_updated (5).csv']


In [ ]:
df_main     = pd.read_csv('dataset_updated.csv')
df_severity = pd.read_csv('Symptom-severity_updated.csv')
df_desc     = pd.read_csv('symptom_Description_updated.csv')
df_prec     = pd.read_csv('symptom_precaution_updated.csv')

df_main['Disease']     = df_main['Disease'].str.strip()
df_severity['Symptom'] = df_severity['Symptom'].str.strip()
df_desc['Disease']     = df_desc['Disease'].str.strip()
df_prec['Disease']     = df_prec['Disease'].str.strip()

symptom_cols = [c for c in df_main.columns if 'Symptom' in c]
for col in symptom_cols:
    df_main[col] = df_main[col].str.strip()

print("4 files loaded successfully!")
print(f"\nMain dataset shape: {df_main.shape}")
print(f" {df_main.shape[0]} patient rows, {df_main.shape[1]} columns")
print(f"\nTotal unique diseases in dataset: {df_main['Disease'].nunique()}")
print(f"\nSeverity file: {df_severity.shape[0]} symptoms with weights")
print(f"Description file: {df_desc.shape[0]} disease descriptions")
print(f"Precaution file: {df_prec.shape[0]} disease precaution sets")

4 files loaded successfully!

Main dataset shape: (4925, 18)
 4925 patient rows, 18 columns

Total unique diseases in dataset: 42

Severity file: 139 symptoms with weights
Description file: 42 disease descriptions
Precaution file: 42 disease precaution sets


In [ ]:
OUR_DISEASES = df_prec['Disease'].unique().tolist()
df = df_main[df_main['Disease'].isin(OUR_DISEASES)].reset_index(drop=True)

print(f"Dataset shape after filter: {df.shape}")
print(f"\nRow count per disease:")
print(df['Disease'].value_counts().to_string())

Dataset shape after filter: (4925, 18)

Row count per disease:
Disease
Fungal infection                           120
Allergy                                    120
GERD                                       120
Chronic cholestasis                        120
Drug Reaction                              120
Peptic ulcer diseae                        120
AIDS                                       120
Diabetes                                   120
Gastroenteritis                            120
Bronchial Asthma                           120
Hypertension                               120
Migraine                                   120
Cervical spondylosis                       120
Paralysis (brain hemorrhage)               120
Jaundice                                   120
Malaria                                    120
Chicken pox                                120
Dengue                                     120
Typhoid                                    120
hepatitis A                         

In [ ]:
print("Missing values per column:")
print(df.isnull().sum().to_string())
print(f"\nDisease column missing values: {df['Disease'].isnull().sum()}")

Missing values per column:
Disease          0
Symptom_1        0
Symptom_2        0
Symptom_3        0
Symptom_4      348
Symptom_5     1206
Symptom_6     1987
Symptom_7     2657
Symptom_8     2981
Symptom_9     3233
Symptom_10    3413
Symptom_11    3731
Symptom_12    4181
Symptom_13    4421
Symptom_14    4619
Symptom_15    4685
Symptom_16    4733
Symptom_17    4853

Disease column missing values: 0


---
## Phase 2 — Exploratory Data Analysis (EDA)

In [ ]:
def format_disease_name(name):
    return str(name).replace('_', ' ').replace('diseae', 'disease').title()

disease_counts = df['Disease'].value_counts()
fig, ax = plt.subplots(figsize=(12, 8))
colors = sns.color_palette("Set2", len(disease_counts))
bars = ax.barh([format_disease_name(d) for d in disease_counts.index],
               disease_counts.values, color=colors, edgecolor='white')
ax.set_xlabel("Number of Patient Records", fontsize=12)
ax.set_title("Patient Records per Disease (All Diseases)", fontsize=14, fontweight='bold')
ax.set_xlim(0, max(disease_counts.values) + 20)
for bar, val in zip(bars, disease_counts.values):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10)
plt.tight_layout()
plt.savefig('plot1_all_diseases_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Total diseases: {df['Disease'].nunique()}")

Total diseases: 42


In [ ]:
all_symptoms = []
for col in symptom_cols:
    vals = df[col].dropna().tolist()
    all_symptoms.extend(vals)

symptom_freq = pd.Series(all_symptoms).value_counts()
top20 = symptom_freq.head(20)

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette("Blues_r", 20)
bars = ax.barh(top20.index[::-1], top20.values[::-1], color=colors[::-1])
ax.set_xlabel("Frequency (number of patient records)", fontsize=12)
ax.set_title("Top 20 Most Frequent Symptoms Across All Diseases", fontsize=14, fontweight='bold')
for bar, val in zip(bars, top20.values[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10)
plt.tight_layout()
plt.savefig('plot2_top_symptoms.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
our_symptom_list = list(pd.Series(all_symptoms).unique())
df_sev_filtered = df_severity[df_severity['Symptom'].isin(our_symptom_list)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_sev_filtered['weight'], bins=7, range=(0.5, 7.5),
             color='steelblue', edgecolor='white', rwidth=0.8)
axes[0].set_xlabel("Severity Weight", fontsize=12)
axes[0].set_ylabel("Number of Symptoms", fontsize=12)
axes[0].set_title("Symptom Severity Weight Distribution", fontsize=12, fontweight='bold')
axes[0].set_xticks(range(1, 8))

top_severe = df_sev_filtered.sort_values('weight', ascending=False).head(15)
colors_sev = ['#d62728' if w >= 6 else '#ff7f0e' if w >= 4 else '#2ca02c'
              for w in top_severe['weight']]
axes[1].barh(top_severe['Symptom'][::-1], top_severe['weight'][::-1], color=colors_sev[::-1])
axes[1].set_xlabel("Severity Weight (1=mild, 7=severe)", fontsize=12)
axes[1].set_title("Top 15 Most Severe Symptoms", fontsize=12, fontweight='bold')
axes[1].set_xlim(0, 8)
plt.tight_layout()
plt.savefig('plot3_severity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
dummies_list = []
for col in symptom_cols:
    dummies = pd.get_dummies(df[col])
    dummies_list.append(dummies)

binary_matrix = pd.concat(dummies_list, axis=1)
binary_matrix = binary_matrix.T.groupby(binary_matrix.columns).max().T
binary_matrix = binary_matrix.fillna(0).astype(int)
binary_matrix['Disease'] = df['Disease'].values

disease_symptom_avg = binary_matrix.groupby('Disease').mean()
top_symptoms_heatmap = disease_symptom_avg.std().sort_values(ascending=False).head(25).index
heatmap_data = disease_symptom_avg[top_symptoms_heatmap]

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=True, fmt='.2f',
            linewidths=0.4,
            cbar_kws={'label': 'Avg Symptom Presence'}, ax=ax)
ax.set_title("Symptom-Disease Co-occurrence Heatmap (Top 25 distinctive symptoms)",
             fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('plot4_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 3 — Preprocessing & Feature Engineering

In [ ]:
dummies_list = []
for col in symptom_cols:
    dummies = pd.get_dummies(df[col])
    dummies_list.append(dummies)

X = pd.concat(dummies_list, axis=1)
X = X.T.groupby(X.columns).max().T
X = X.fillna(0).astype(int)

print(" Binary feature matrix X created!")
print(f" Shape: {X.shape}  ({X.shape[0]} patients × {X.shape[1]} symptoms)")

 Binary feature matrix X created!
 Shape: (4925, 138)  (4925 patients × 138 symptoms)


In [ ]:
le = LabelEncoder()
y = le.fit_transform(df['Disease'])
NUM_CLASSES = len(le.classes_)

print(f"Labels encoded! Total disease classes: {NUM_CLASSES}")
print("\nEncoding map:")
for i, cls in enumerate(le.classes_):
    print(f"  {i:2d}  →  {cls}")

Labels encoded! Total disease classes: 42

Encoding map:
   0  →  (vertigo) Paroymsal  Positional Vertigo
   1  →  AIDS
   2  →  Acne
   3  →  Alcoholic hepatitis
   4  →  Allergy
   5  →  Arthritis
   6  →  Bronchial Asthma
   7  →  Cervical spondylosis
   8  →  Chicken pox
   9  →  Chronic cholestasis
  10  →  Common Cold
  11  →  Dengue
  12  →  Diabetes
  13  →  Dimorphic hemmorhoids(piles)
  14  →  Drug Reaction
  15  →  Fungal infection
  16  →  GERD
  17  →  Gastroenteritis
  18  →  Heart attack
  19  →  Hepatitis B
  20  →  Hepatitis C
  21  →  Hepatitis D
  22  →  Hepatitis E
  23  →  Hypertension
  24  →  Hyperthyroidism
  25  →  Hypoglycemia
  26  →  Hypothyroidism
  27  →  Impetigo
  28  →  Jaundice
  29  →  Malaria
  30  →  Migraine
  31  →  Osteoarthristis
  32  →  PCOS
  33  →  Paralysis (brain hemorrhage)
  34  →  Peptic ulcer diseae
  35  →  Pneumonia
  36  →  Psoriasis
  37  →  Tuberculosis
  38  →  Typhoid
  39  →  Urinary tract infection
  40  →  Varicose veins
  41

In [ ]:
df_severity['Symptom'] = df_severity['Symptom'].str.strip()
severity_dict = dict(zip(df_severity['Symptom'], df_severity['weight']))

def calculate_severity_score(row):
    total = 0
    for col in symptom_cols:
        symptom = row[col]
        if pd.notna(symptom) and str(symptom).strip() != '':
            total += severity_dict.get(str(symptom).strip(), 0)
    return total

df['severity_score'] = df.apply(calculate_severity_score, axis=1)
df['severity_score'] = pd.to_numeric(df['severity_score'], errors='coerce').fillna(0)
df['risk_level'] = pd.cut(df['severity_score'],
                          bins=[0, 13, 26, float('inf')],
                          labels=['Low', 'Medium', 'High'],
                          include_lowest=True)
X['severity_score'] = df['severity_score'].values

print("Severity scores and risk levels calculated!")
print(f"\nSeverity score stats:")
print(df['severity_score'].describe().round(2).to_string())
print(f"\nRisk level distribution:")
print(df['risk_level'].value_counts().to_string())

Severity scores and risk levels calculated!

Severity score stats:
count    4925.00
mean       30.72
std        16.54
min         4.00
25%        18.00
50%        27.00
75%        42.00
max        77.00

Risk level distribution:
risk_level
High      2544
Medium    1901
Low        480


In [ ]:
np.random.seed(42)
X_noisy = X.copy().astype(float)

symptom_feature_cols = [c for c in X_noisy.columns if c != 'severity_score']
for col in symptom_feature_cols:
    flip_mask = np.random.rand(len(X_noisy)) < 0.03
    X_noisy.loc[flip_mask, col] = 1 - X_noisy.loc[flip_mask, col]
X_noisy['severity_score'] += np.random.normal(0, 2.0, len(X_noisy))

X_train, X_test, y_train, y_test = train_test_split(
    X_noisy.values, y, test_size=0.2, random_state=42, stratify=y)

print("Data split into training and test sets (with noise augmentation)!")
print(f"   Training set : {X_train.shape[0]} rows")
print(f"   Test set     : {X_test.shape[0]} rows")
print(f"   Features     : {X_train.shape[1]}")

Data split into training and test sets (with noise augmentation)!
   Training set : 3940 rows
   Test set     : 985 rows
   Features     : 139


---
## Phase 4 — Machine Learning & Deep Learning Models

We train **8 models**  
Decision Tree → Random Forest → Logistic Regression → SVM → Linear Regression → ANN → K-Means → CNN

### Phase 4A — Decision Tree Classifier

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)
dt_acc = accuracy_score(y_test, y_pred_dt)

print("Decision Tree trained!")
print(f"Accuracy on test set: {dt_acc*100:.2f}%")

Decision Tree trained!
Accuracy on test set: 75.84%


### Phase 4B — Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=3,
                                   max_features=0.6, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)

print("Random Forest trained!")
print(f"Accuracy on test set: {rf_acc*100:.2f}%")
print(f"Decision Tree : {dt_acc*100:.2f}%")
print(f"Random Forest : {rf_acc*100:.2f}%")

Random Forest trained!
Accuracy on test set: 89.75%
Decision Tree : 75.84%
Random Forest : 89.75%


### Phase 4C — Logistic Regression

In [ ]:
# ─────────────────────────────────────────────────────────────
# LOGISTIC REGRESSION
# A probabilistic linear classifier.
# For multi-class (42 diseases) we use multinomial softmax.
# Works well when features are linearly separable.
# ─────────────────────────────────────────────────────────────

# Scale features — Logistic Regression converges faster with normalized input
scaler_lr = StandardScaler()
X_train_lr = scaler_lr.fit_transform(X_train)
X_test_lr  = scaler_lr.transform(X_test)

lr_model = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train_lr, y_train)

y_pred_lr = lr_model.predict(X_test_lr)
lr_acc = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression trained!")
print(f"Accuracy on test set: {lr_acc*100:.2f}%")
print(f"\nInsight: Logistic Regression uses a softmax function across all {NUM_CLASSES} disease classes.")
print(f"It outputs a probability distribution — the disease with highest probability wins.")

Logistic Regression trained!
Accuracy on test set: 99.29%

Insight: Logistic Regression uses a softmax function across all 42 disease classes.
It outputs a probability distribution — the disease with highest probability wins.


### Phase 4D — Support Vector Machine (SVM)

In [ ]:
# ─────────────────────────────────────────────────────────────
# SUPPORT VECTOR MACHINE (SVM)
# Finds the optimal hyperplane that maximally separates classes.
# RBF kernel maps data to higher dimensions to find non-linear boundaries.
# Particularly effective on high-dimensional binary feature spaces like ours.
# ─────────────────────────────────────────────────────────────

scaler_svm = StandardScaler()
X_train_svm = scaler_svm.fit_transform(X_train)
X_test_svm  = scaler_svm.transform(X_test)

svm_model = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    probability=True,       # enable predict_proba for top-3 predictions
    random_state=42
)
svm_model.fit(X_train_svm, y_train)

y_pred_svm = svm_model.predict(X_test_svm)
svm_acc = accuracy_score(y_test, y_pred_svm)

print("SVM (RBF kernel) trained!")
print(f"Accuracy on test set: {svm_acc*100:.2f}%")
print(f"\nInsight: RBF kernel allows SVM to handle non-linear decision boundaries.")
print(f"C=10 means we allow minimal margin violations (low tolerance for misclassification).")

SVM (RBF kernel) trained!
Accuracy on test set: 99.29%

Insight: RBF kernel allows SVM to handle non-linear decision boundaries.
C=10 means we allow minimal margin violations (low tolerance for misclassification).


### Phase 4E — Linear Regression (Severity Score Prediction)
> **New sub-task:** Instead of classifying a disease, we now *predict the severity score* of a patient using their symptoms. This reframes part of the problem as a regression task.

In [ ]:
# ─────────────────────────────────────────────────────────────
# LINEAR REGRESSION — Predict Severity Score from Symptoms
# Input  : Binary symptom vector (54 features)
# Output : Numerical severity score (continuous)
# ─────────────────────────────────────────────────────────────

# Use symptom-only features (exclude severity_score itself)
X_reg = X[symptom_feature_cols].values
y_reg = df['severity_score'].values

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

linreg_model = LinearRegression()
linreg_model.fit(X_train_reg, y_train_reg)

y_pred_reg = linreg_model.predict(X_test_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2   = r2_score(y_test_reg, y_pred_reg)

print("Linear Regression trained for Severity Score Prediction!")
print(f"RMSE (Root Mean Squared Error) : {rmse:.4f}")
print(f"R² Score                       : {r2:.4f}")
print(f"\nInterpretation:")
print(f"R²={r2:.2f} means the model explains {r2*100:.1f}% of the variance in severity scores.")
print(f"RMSE={rmse:.2f} means predictions are off by ±{rmse:.2f} severity points on average.")

# Plot: Actual vs Predicted severity score
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_test_reg, y_pred_reg, alpha=0.4, color='steelblue', edgecolors='white', s=40)
lims = [min(y_test_reg.min(), y_pred_reg.min()),
        max(y_test_reg.max(), y_pred_reg.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel("Actual Severity Score", fontsize=12)
ax.set_ylabel("Predicted Severity Score", fontsize=12)
ax.set_title(f"Linear Regression — Severity Prediction\nRMSE={rmse:.2f}  R²={r2:.3f}",
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('plot_linreg_severity.png', dpi=150, bbox_inches='tight')
plt.show()

Linear Regression trained for Severity Score Prediction!
RMSE (Root Mean Squared Error) : 0.0000
R² Score                       : 1.0000

Interpretation:
R²=1.00 means the model explains 100.0% of the variance in severity scores.
RMSE=0.00 means predictions are off by ±0.00 severity points on average.


### Phase 4F — Artificial Neural Network (ANN)
> **Using TensorFlow + Keras.**  
> Architecture: Input(55) → Dense(256, ReLU) → Dropout(0.4) → Dense(128, ReLU) → Dropout(0.3) → Dense(64, ReLU) → Output(N_classes, Softmax)

In [ ]:
# ─────────────────────────────────────────────────────────────
# ARTIFICIAL NEURAL NETWORK (ANN)
# A fully-connected feedforward network (Multi-Layer Perceptron).
# Dropout layers prevent overfitting.
# Softmax output gives probability over all disease classes.
# ─────────────────────────────────────────────────────────────

# One-hot encode labels for ANN (required by categorical_crossentropy)
y_train_cat = to_categorical(y_train, num_classes=NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  num_classes=NUM_CLASSES)

# Normalize input (mean=0, std=1) — critical for neural networks
scaler_ann = StandardScaler()
X_train_ann = scaler_ann.fit_transform(X_train)
X_test_ann  = scaler_ann.transform(X_test)

def build_ann(input_dim, num_classes):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),

        # Hidden Layer 1
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),

        # Hidden Layer 2
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        # Hidden Layer 3
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),

        # Output Layer
        layers.Dense(num_classes, activation='softmax')
    ], name='CareBridge_ANN')
    return model

ann_model = build_ann(X_train_ann.shape[1], NUM_CLASSES)
ann_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=15,
                            restore_best_weights=True, verbose=1)

ann_history = ann_model.fit(
    X_train_ann, y_train_cat,
    validation_split=0.15,
    epochs=150,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

y_pred_ann_proba = ann_model.predict(X_test_ann)
y_pred_ann = np.argmax(y_pred_ann_proba, axis=1)
ann_acc = accuracy_score(y_test, y_pred_ann)

print(f"\nANN trained!")
print(f"Accuracy on test set: {ann_acc*100:.2f}%")

Model: "CareBridge_ANN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 256)            │        35,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 42)             │         2,730 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81,258 (317.41 KB)

 Trainable params: 80,490 (314.41 KB)

 Non-trainable params: 768 (3.00 KB)

Epoch 1/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.3320 - loss: 2.7460 - val_accuracy: 0.8765 - val_loss: 1.4830
Epoch 2/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8071 - loss: 0.9679 - val_accuracy: 0.9763 - val_loss: 0.2766
Epoch 3/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9071 - loss: 0.4357 - val_accuracy: 0.9848 - val_loss: 0.0931
Epoch 4/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9379 - loss: 0.2699 - val_accuracy: 0.9898 - val_loss: 0.0526
Epoch 5/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9552 - loss: 0.1885 - val_accuracy: 0.9915 - val_loss: 0.0386
Epoch 6/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9669 - loss: 0.1383 - val_accuracy: 0.9915 - val_loss: 0.0323
Epoch 7/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9722 - loss: 0.1096 - val_accuracy: 0.9966 - val_loss: 0.0260
Epoch 8/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9770 - loss: 0.0934 - val_accu

In [ ]:
# ─── Training Curves (Loss & Accuracy over Epochs) ───────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(ann_history.history['loss'],     label='Train Loss',      color='steelblue')
axes[0].plot(ann_history.history['val_loss'], label='Validation Loss', color='crimson', linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('ANN Training — Loss Curve', fontweight='bold')
axes[0].legend()

# Accuracy curve
axes[1].plot(ann_history.history['accuracy'],     label='Train Accuracy',      color='steelblue')
axes[1].plot(ann_history.history['val_accuracy'], label='Validation Accuracy', color='crimson', linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('ANN Training — Accuracy Curve', fontweight='bold')
axes[1].legend()

plt.suptitle('ANN Training Curves (CareBridge)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_ann_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### Phase 4G — Unsupervised Learning: K-Means Clustering + PCA
> **Research question:** *Can a machine discover disease families on its own — without being told any labels?*  
> We apply K-Means to the symptom vectors and then use PCA to visualize the clusters in 2D.

In [ ]:
# ─────────────────────────────────────────────────────────────
# K-MEANS CLUSTERING
# Groups patients purely based on their symptom patterns.
# No disease labels are used — purely unsupervised.
# We then check if discovered clusters align with actual diseases.
# ─────────────────────────────────────────────────────────────

# Use only symptom features (no severity_score)
X_cluster = X[symptom_feature_cols].values.astype(float)

# ── Step 1: Elbow Method — find optimal K ─────────────────────
inertias = []
K_range  = range(2, 20)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, inertias, 'bo-', linewidth=2, markersize=6)
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia (Within-cluster Sum of Squares)')
ax.set_title('Elbow Method — Optimal K for Disease Clustering', fontweight='bold')
ax.axvline(x=10, color='red', linestyle='--', alpha=0.7, label='K=10 (chosen)')
ax.legend()
plt.tight_layout()
plt.savefig('plot_kmeans_elbow.png', dpi=150, bbox_inches='tight')
plt.show()
print("Elbow curve plotted. We choose K=10 (matching disease family count).")

Elbow curve plotted. We choose K=10 (matching disease family count).


In [ ]:
# ── Step 2: Fit K-Means with K=10 ─────────────────────────────
OPTIMAL_K = 10
kmeans_model = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
cluster_labels = kmeans_model.fit_predict(X_cluster)

print(f"K-Means fitted with K={OPTIMAL_K}")
print(f"Cluster sizes:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for c, cnt in zip(unique, counts):
    print(f"Cluster {c}: {cnt} patients")

K-Means fitted with K=10
Cluster sizes:
Cluster 0: 240 patients
Cluster 1: 1355 patients
Cluster 2: 120 patients
Cluster 3: 1290 patients
Cluster 4: 240 patients
Cluster 5: 120 patients
Cluster 6: 474 patients
Cluster 7: 240 patients
Cluster 8: 486 patients
Cluster 9: 360 patients


In [ ]:
# ── Step 3: PCA — Reduce 54 dims to 2D for visualization ───────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster)

explained = pca.explained_variance_ratio_
print(f"PCA Explained Variance: PC1={explained[0]*100:.1f}%  PC2={explained[1]*100:.1f}%")
print(f"Total variance captured by 2 components: {sum(explained)*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: K-Means cluster assignments
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                            c=cluster_labels, cmap='tab10', alpha=0.5, s=15)
axes[0].set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)')
axes[0].set_title('K-Means Clusters (Unsupervised)\nPatients grouped purely by symptoms',
                  fontweight='bold')
plt.colorbar(scatter1, ax=axes[0], label='Cluster ID')

# Plot 2: Actual disease labels (for comparison)
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                            c=y, cmap='tab20', alpha=0.5, s=15)
axes[1].set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)')
axes[1].set_title('Actual Disease Labels (Supervised)\nColoured by true disease class',
                  fontweight='bold')
plt.colorbar(scatter2, ax=axes[1], label='Disease Class')

plt.suptitle('PCA Visualization — K-Means Clusters vs Actual Disease Labels',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_kmeans_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nInsight: Overlapping regions show diseases with similar symptom profiles.")
print("Well-separated clusters (e.g. Hepatitis variants) show unique symptom patterns.")

In [ ]:
# ── Step 4: Cluster vs Disease Analysis ────────────────────────
# See which diseases are grouped into which clusters
df_cluster_analysis = pd.DataFrame({
    'Disease' : df['Disease'].values,
    'Cluster' : cluster_labels
})

print("K-Means Cluster → Disease Breakdown:")
print("="*60)
for cluster_id in range(OPTIMAL_K):
    subset = df_cluster_analysis[df_cluster_analysis['Cluster'] == cluster_id]
    top_diseases = subset['Disease'].value_counts().head(3)
    diseases_str = ', '.join([f"{d}({n})" for d, n in top_diseases.items()])
    print(f"  Cluster {cluster_id:2d}: {diseases_str}")

print("\nObservation: Clusters often group medically related diseases together")
print("(e.g., Hepatitis A/B/C/D/E in one cluster, skin diseases in another).")

### Phase 4H — CNN (Convolutional Neural Network) on Symptom Grid
> **Concept:** We reshape each patient's 54 binary symptom features into a **2D grid (6×9)**  
> and treat it like a single-channel image. A LeNet-inspired CNN then learns spatial/co-occurrence  
> patterns between neighboring symptoms — analogous to how CNNs detect edges in image patches.

**CNN Architecture used:** LeNet-5 inspired  
Input(6×9×1) → Conv2D(32,3×3,ReLU) → MaxPool → Conv2D(64,3×3,ReLU) → MaxPool → Flatten → Dense(128,ReLU) → Dropout → Dense(N,Softmax)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CNN — Symptom Grid Representation
#
# Why CNN on tabular data?
# When symptoms are arranged spatially, CNN convolutional filters
# can detect local co-occurrence patterns (e.g., fever+chills often
# appear together). This is a creative and academically valid
# approach to apply CNN to non-image healthcare data.
#
# Architecture family: LeNet-5 inspired (LeCun et al., 1998)
# ─────────────────────────────────────────────────────────────

# ── Step 1: Prepare symptom-only features ─────────────────────
# We use X_noisy without severity_score for CNN
X_cnn_raw = X_noisy[symptom_feature_cols].values
NUM_SYMPTOMS = X_cnn_raw.shape[1]   # e.g. 54

# Pad to next perfect grid size (6×9 = 54, or 7×8=56, etc.)
GRID_H = 6
GRID_W = 9
GRID_SIZE = GRID_H * GRID_W  # 54

# Pad or trim features to fit exactly into the grid
if NUM_SYMPTOMS < GRID_SIZE:
    pad_width = GRID_SIZE - NUM_SYMPTOMS
    X_cnn_raw = np.hstack([X_cnn_raw, np.zeros((X_cnn_raw.shape[0], pad_width))])
else:
    X_cnn_raw = X_cnn_raw[:, :GRID_SIZE]

# Reshape to (samples, height, width, channels)
X_cnn = X_cnn_raw.reshape(-1, GRID_H, GRID_W, 1).astype(np.float32)

# Train-test split
X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(
    X_cnn, y, test_size=0.2, random_state=42, stratify=y)

# One-hot encode
y_train_cnn_cat = to_categorical(y_train_cnn, num_classes=NUM_CLASSES)
y_test_cnn_cat  = to_categorical(y_test_cnn,  num_classes=NUM_CLASSES)

print(f"Symptom Grid prepared for CNN!")
print(f"Input shape per sample : ({GRID_H}, {GRID_W}, 1) — a {GRID_H}×{GRID_W} single-channel grid")
print(f"Training samples       : {X_train_cnn.shape[0]}")
print(f"Test samples           : {X_test_cnn.shape[0]}")
print(f"Output classes         : {NUM_CLASSES} diseases")

In [ ]:
# ── Visualize the symptom grid for a sample patient ───────────
symptom_names_grid = symptom_feature_cols[:GRID_SIZE]

sample_idx = 0
sample_grid = X_train_cnn[sample_idx, :, :, 0]
sample_disease = le.inverse_transform([y_train_cnn[sample_idx]])[0]

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(sample_grid, cmap='Greens', vmin=0, vmax=1, aspect='auto')
ax.set_title(f'Symptom Grid Representation — Patient with: {sample_disease}\n'
             f'(Green=symptom present, White=absent) — {GRID_H}×{GRID_W} grid',
             fontweight='bold')
ax.set_xlabel('Symptom Index (column)')
ax.set_ylabel('Symptom Group (row)')

# Add cell annotations
for i in range(GRID_H):
    for j in range(GRID_W):
        idx = i * GRID_W + j
        if idx < len(symptom_names_grid):
            sname = symptom_names_grid[idx].replace('_', ' ')[:8]
            color = 'white' if sample_grid[i, j] > 0.5 else 'gray'
            ax.text(j, i, sname, ha='center', va='center',
                    fontsize=6, color=color)
plt.colorbar(im, ax=ax, label='Symptom Presence')
plt.tight_layout()
plt.savefig('plot_cnn_symptom_grid.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Build LeNet-inspired CNN ───────────────────────────────────
def build_lenet_cnn(input_shape, num_classes):
    """
    LeNet-5 inspired CNN adapted for symptom grid input.
    Reference: LeCun, Y. et al. (1998). Gradient-based learning
               applied to document recognition.
    """
    model = models.Sequential([
        # C1: First Conv Layer (like LeNet C1)
        layers.Conv2D(32, kernel_size=(2, 3), activation='relu',
                      padding='same', input_shape=input_shape,
                      name='C1_Conv'),
        layers.BatchNormalization(name='C1_BN'),
        # S2: Pooling (like LeNet S2)
        layers.MaxPooling2D(pool_size=(1, 2), name='S2_Pool'),

        # C3: Second Conv Layer (like LeNet C3)
        layers.Conv2D(64, kernel_size=(2, 2), activation='relu',
                      padding='same', name='C3_Conv'),
        layers.BatchNormalization(name='C3_BN'),
        # S4: Pooling (like LeNet S4)
        layers.MaxPooling2D(pool_size=(1, 2), name='S4_Pool'),

        # C5: Third Conv Layer (like LeNet C5)
        layers.Conv2D(128, kernel_size=(2, 2), activation='relu',
                      padding='same', name='C5_Conv'),

        # Flatten
        layers.Flatten(name='Flatten'),

        # F6: Fully Connected (like LeNet F6)
        layers.Dense(256, activation='relu', name='F6_Dense'),
        layers.Dropout(0.5, name='F6_Dropout'),

        layers.Dense(128, activation='relu', name='F7_Dense'),
        layers.Dropout(0.3, name='F7_Dropout'),

        # Output
        layers.Dense(num_classes, activation='softmax', name='Output')
    ], name='CareBridge_LeNet_CNN')
    return model

cnn_model = build_lenet_cnn(
    input_shape=(GRID_H, GRID_W, 1),
    num_classes=NUM_CLASSES
)
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
# ── Train CNN ──────────────────────────────────────────────────
early_stop_cnn = EarlyStopping(monitor='val_loss', patience=20,
                                restore_best_weights=True, verbose=1)

cnn_history = cnn_model.fit(
    X_train_cnn, y_train_cnn_cat,
    validation_split=0.15,
    epochs=150,
    batch_size=32,
    callbacks=[early_stop_cnn],
    verbose=1
)

y_pred_cnn_proba = cnn_model.predict(X_test_cnn)
y_pred_cnn = np.argmax(y_pred_cnn_proba, axis=1)
cnn_acc = accuracy_score(y_test_cnn, y_pred_cnn)

print(f"\nLeNet-inspired CNN trained!")
print(f"Accuracy on test set: {cnn_acc*100:.2f}%")

In [ ]:
# ── CNN Training Curves ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(cnn_history.history['loss'],     label='Train Loss',      color='darkorange')
axes[0].plot(cnn_history.history['val_loss'], label='Validation Loss', color='navy', linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('CNN Training — Loss Curve', fontweight='bold')
axes[0].legend()

axes[1].plot(cnn_history.history['accuracy'],     label='Train Accuracy',      color='darkorange')
axes[1].plot(cnn_history.history['val_accuracy'], label='Validation Accuracy', color='navy', linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('CNN Training — Accuracy Curve', fontweight='bold')
axes[1].legend()

plt.suptitle('LeNet-inspired CNN Training Curves (CareBridge)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_cnn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
cm_cnn = confusion_matrix(y_test_cnn, y_pred_cnn)
labels = [format_disease_name(c) for c in le.classes_]
n = len(labels)

fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(cm_cnn, interpolation='nearest', cmap='Oranges')

ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))
ax.set_xticklabels(labels, rotation=90, fontsize=5)
ax.set_yticklabels(labels, fontsize=5)

for i in range(n):
    val = cm_cnn[i, i]
    if val > 0:
        ax.text(i, i, str(val), ha='center', va='center',
                fontsize=4, color='black', fontweight='bold')

ax.set_xlabel('Predicted Label', fontsize=10)
ax.set_ylabel('True Label', fontsize=10)
ax.set_title('Confusion Matrix - LeNet CNN (CareBridge)', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
plt.tight_layout()
plt.savefig('plot_cnn_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()


### Phase 4I — Comprehensive Model Comparison (All 6 Classifiers)

In [ ]:
# ─────────────────────────────────────────────────────────────
# COMPARATIVE ANALYSIS — All 6 Classification Models
# ─────────────────────────────────────────────────────────────

model_results = {
    'Decision Tree'       : dt_acc  * 100,
    'Random Forest'       : rf_acc  * 100,
    'Logistic Regression' : lr_acc  * 100,
    'SVM (RBF)'           : svm_acc * 100,
    'ANN (Keras)'         : ann_acc * 100,
    'CNN (LeNet)'         : cnn_acc * 100,
}

results_df = pd.DataFrame(list(model_results.items()), columns=['Model', 'Accuracy (%)'])
results_df = results_df.sort_values('Accuracy (%)', ascending=True).reset_index(drop=True)

# Color-code: best = green, worst = red
max_acc = results_df['Accuracy (%)'].max()
bar_colors = ['#2ecc71' if acc == max_acc else '#3498db' if acc >= max_acc - 5 else '#e74c3c'
              for acc in results_df['Accuracy (%)']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(results_df['Model'], results_df['Accuracy (%)'],
               color=bar_colors, edgecolor='white', height=0.6)

for bar, acc in zip(bars, results_df['Accuracy (%)']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{acc:.2f}%', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Accuracy (%)', fontsize=12)
ax.set_title('CareBridge — Model Accuracy Comparison\n(All AI/ML Models)', fontsize=14, fontweight='bold')
ax.set_xlim(0, max_acc + 10)
ax.axvline(x=results_df['Accuracy (%)'].mean(), color='orange',
           linestyle='--', linewidth=1.5, label=f'Mean Accuracy: {results_df["Accuracy (%)"].mean():.1f}%')
ax.legend()

plt.tight_layout()
plt.savefig('plot_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*55)
print("  MODEL COMPARISON TABLE")
print("="*55)
print(f"  {'Model':<25} {'Accuracy':>10}  {'Type'}")
print("-"*55)
model_types = {
    'Decision Tree'       : 'Supervised / Tree',
    'Random Forest'       : 'Supervised / Ensemble',
    'Logistic Regression' : 'Supervised / Linear',
    'SVM (RBF)'           : 'Supervised / Kernel',
    'ANN (Keras)'         : 'Deep Learning / MLP',
    'CNN (LeNet)'         : 'Deep Learning / CNN',
}
for model, acc in sorted(model_results.items(), key=lambda x: -x[1]):
    mtype = model_types[model]
    best  = " ← BEST" if acc == max_acc else ""
    print(f"  {model:<25} {acc:>8.2f}%  {mtype}{best}")
print("="*55)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 14))
for ax, preds, title, cmap in zip(
    axes,
    [y_pred_dt, y_pred_rf],
    ['Decision Tree', 'Random Forest'],
    ['Blues', 'Blues']
):
    cm = confusion_matrix(y_test, preds)
    labels = [format_disease_name(c) for c in le.classes_]
    n = len(labels)
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(labels, rotation=90, fontsize=5)
    ax.set_yticklabels(labels, fontsize=5)
    for i in range(n):
        val = cm[i, i]
        if val > 0:
            ax.text(i, i, str(val), ha='center', va='center', fontsize=4, color='black', fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=9)
    ax.set_ylabel('True Label', fontsize=9)
    ax.set_title(f'Confusion Matrix - {title}', fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
plt.tight_layout()
plt.savefig('plot5_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(16, 14))
cm_ann = confusion_matrix(y_test, y_pred_ann)
labels = [format_disease_name(c) for c in le.classes_]
n = len(labels)
im = ax.imshow(cm_ann, interpolation='nearest', cmap='Greens')
ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))
ax.set_xticklabels(labels, rotation=90, fontsize=5)
ax.set_yticklabels(labels, fontsize=5)
for i in range(n):
    val = cm_ann[i, i]
    if val > 0:
        ax.text(i, i, str(val), ha='center', va='center', fontsize=4, color='black', fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=10)
ax.set_ylabel('True Label', fontsize=10)
ax.set_title('Confusion Matrix - ANN (Keras)', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
plt.tight_layout()
plt.savefig('plot_ann_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Feature Importance (Random Forest) ────────────────────────
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top15_features = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 7))
colors = sns.color_palette("viridis", 15)
top15_features[::-1].plot(kind='barh', ax=ax, color=colors[::-1])
ax.set_title("Top 15 Most Important Symptoms (Random Forest)", fontsize=13, fontweight='bold')
ax.set_xlabel("Feature Importance Score", fontsize=12)
for i, v in enumerate(top15_features.values[::-1]):
    ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('plot6_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 5 — Health Risk Assessment & Department Recommendation
> The `carebridge_predict()` function now uses the **best-performing model** automatically.

In [ ]:
DEPARTMENT_MAP = {
    'Diabetes'                            : ('Diabetes (Type 2)', 'Endocrinology'),
    'Hypothyroidism'                      : ('Thyroid Disorder', 'Endocrinology'),
    'Hyperthyroidism'                     : ('Thyroid Disorder', 'Endocrinology'),
    'Hypoglycemia'                        : ('Blood Sugar Disorder', 'Endocrinology'),
    'Hypertension'                        : ('Hypertension', 'Cardiology'),
    'Heart attack'                        : ('Heart Disease', 'Cardiology'),
    'Acne'                                : ('Skin Condition', 'Dermatology'),
    'Psoriasis'                           : ('Skin Condition', 'Dermatology'),
    'Impetigo'                            : ('Skin Infection', 'Dermatology'),
    'GERD'                                : ('Acidity / GERD', 'Gastroenterology'),
    'Peptic ulcer diseae'                 : ('Peptic Ulcer', 'Gastroenterology'),
    'Gastroenteritis'                     : ('Stomach Infection', 'Gastroenterology'),
    'Chronic cholestasis'                 : ('Liver Disorder', 'Gastroenterology'),
    'Jaundice'                            : ('Liver Disorder', 'Gastroenterology'),
    'Migraine'                            : ('Migraine / Headache', 'Neurology'),
    '(vertigo) Paroymsal Positional Vertigo': ('Vertigo', 'Neurology'),
    'Paralysis (brain hemorrhage)'        : ('Neurological Disorder', 'Neurology'),
    'Bronchial Asthma'                    : ('Asthma', 'Pulmonology'),
    'Pneumonia'                           : ('Lung Infection', 'Pulmonology'),
    'Tuberculosis'                        : ('Lung Infection', 'Pulmonology'),
    'Malaria'                             : ('Vector-borne Disease', 'Infectious Diseases'),
    'Dengue'                              : ('Vector-borne Disease', 'Infectious Diseases'),
    'Typhoid'                             : ('Bacterial Infection', 'Infectious Diseases'),
    'Chicken pox'                         : ('Viral Infection', 'Infectious Diseases'),
    'Common Cold'                         : ('Viral Infection', 'Infectious Diseases'),
    'Hepatitis A'                         : ('Liver Infection', 'Infectious Diseases'),
    'Hepatitis B'                         : ('Liver Infection', 'Infectious Diseases'),
    'Hepatitis C'                         : ('Liver Infection', 'Infectious Diseases'),
    'Hepatitis D'                         : ('Liver Infection', 'Infectious Diseases'),
    'Hepatitis E'                         : ('Liver Infection', 'Infectious Diseases'),
    'AIDS'                                : ('Immunodeficiency', 'Infectious Diseases'),
    'Arthritis'                           : ('Joint Disorder', 'Orthopedics'),
    'Osteoarthritis'                      : ('Joint Disorder', 'Orthopedics'),
    'Cervical spondylosis'                : ('Spine Disorder', 'Orthopedics'),
    'Urinary tract infection'             : ('UTI', 'Urology'),
    'Varicose veins'                      : ('Vascular Disorder', 'Cardiology'),
    'Allergy'                             : ('Allergic Condition', 'General Medicine'),
    'Dimorphic hemmorhoids(piles)'        : ('Piles', 'General Surgery'),
    'Drug Reaction'                       : ('Adverse Drug Reaction', 'General Medicine'),
    'Fungal infection'                    : ('Fungal Infection', 'Dermatology'),
    'Alcoholic hepatitis'                 : ('Liver Disease', 'Gastroenterology'),
    'PCOS'                                : ('Hormonal Disorder', 'Gynecology'),
}

URGENCY_MSG = {
    'Low'    : 'LOW RISK   — Monitor symptoms at home. Schedule a routine check-up.',
    'Medium' : "MEDIUM RISK — See a doctor within the next few days. Don't delay.",
    'High'   : 'HIGH RISK   — Seek medical attention promptly. Visit the department soon.',
}

desc_dict = dict(zip(df_desc['Disease'], df_desc['Description']))
prec_dict = {}
for _, row in df_prec.iterrows():
    disease = row['Disease']
    precs = [row[f'Precaution_{i}'] for i in range(1, 5)
             if pd.notna(row.get(f'Precaution_{i}', None))]
    prec_dict[disease] = precs

# Determine best classifier for the prediction function
best_model_name = max(model_results, key=model_results.get)
print(f"   Department knowledge base built!")
print(f"   Departments mapped : {len(DEPARTMENT_MAP)} diseases")
print(f"   Descriptions loaded: {len(desc_dict)}")
print(f"   Best model selected: {best_model_name} ({model_results[best_model_name]:.2f}%)")

In [ ]:
def carebridge_predict(symptom_list, patient_name="Patient"):
    """
    CareBridge prediction function.
    Uses the best-performing model (determined by Phase 4I comparison).
    Returns: dict with prediction, confidence, risk level, department.
    """
    print("\n" + "═"*60)
    print(f"  CareBridge Health Assessment Report")
    print(f"  Patient: {patient_name}")
    print("═"*60)

    all_valid_symptoms = set(X.columns) - {'severity_score'}
    valid_symptoms, invalid_symptoms = [], []

    for s in symptom_list:
        s_clean = s.strip().lower().replace(' ', '_')
        if s_clean in all_valid_symptoms:
            valid_symptoms.append(s_clean)
        else:
            invalid_symptoms.append(s)

    if invalid_symptoms:
        print(f"\n⚠ Unknown symptoms (skipped): {invalid_symptoms}")

    if len(valid_symptoms) == 0:
        print("No valid symptoms entered. Cannot make a prediction.")
        return None

    print(f"\nSymptoms entered ({len(valid_symptoms)} recognized):")
    for s in valid_symptoms:
        weight = severity_dict.get(s, 0)
        print(f"    • {s.replace('_', ' ').title():40s} (severity weight: {weight})")

    # ── Build feature vector ───────────────────────────────────
    feature_vector = pd.DataFrame(0, index=[0], columns=X.columns)
    for s in valid_symptoms:
        if s in feature_vector.columns:
            feature_vector[s] = 1

    sev_score = sum(severity_dict.get(s, 0) for s in valid_symptoms)
    feature_vector['severity_score'] = sev_score
    fv = feature_vector.values

    # ── All model predictions ──────────────────────────────────
    fv_scaled_lr  = scaler_lr.transform(fv)
    fv_scaled_svm = scaler_svm.transform(fv)
    fv_scaled_ann = scaler_ann.transform(fv)

    pred_dt  = le.inverse_transform(dt_model.predict(fv))[0]
    pred_rf  = le.inverse_transform(rf_model.predict(fv))[0]
    pred_lr  = le.inverse_transform(lr_model.predict(fv_scaled_lr))[0]
    pred_svm = le.inverse_transform(svm_model.predict(fv_scaled_svm))[0]

    ann_proba   = ann_model.predict(fv_scaled_ann, verbose=0)[0]
    ann_pred_idx= np.argmax(ann_proba)
    pred_ann    = le.inverse_transform([ann_pred_idx])[0]

    # CNN input
    fv_symptom_only = feature_vector[symptom_feature_cols].values
    if fv_symptom_only.shape[1] < GRID_SIZE:
        pad = np.zeros((1, GRID_SIZE - fv_symptom_only.shape[1]))
        fv_symptom_only = np.hstack([fv_symptom_only, pad])
    else:
        fv_symptom_only = fv_symptom_only[:, :GRID_SIZE]
    fv_cnn      = fv_symptom_only.reshape(1, GRID_H, GRID_W, 1).astype(np.float32)
    cnn_proba   = cnn_model.predict(fv_cnn, verbose=0)[0]
    cnn_pred_idx= np.argmax(cnn_proba)
    pred_cnn    = le.inverse_transform([cnn_pred_idx])[0]

    # Use best model for primary prediction
    primary_map = {
        'Decision Tree'       : (pred_dt,  None),
        'Random Forest'       : (pred_rf,  rf_model.predict_proba(fv)[0]),
        'Logistic Regression' : (pred_lr,  lr_model.predict_proba(fv_scaled_lr)[0]),
        'SVM (RBF)'           : (pred_svm, svm_model.predict_proba(fv_scaled_svm)[0]),
        'ANN (Keras)'         : (pred_ann, ann_proba),
        'CNN (LeNet)'         : (pred_cnn, cnn_proba),
    }
    disease_name, probabilities = primary_map[best_model_name]
    if probabilities is not None:
        confidence = np.max(probabilities) * 100
        top3_idx   = np.argsort(probabilities)[::-1][:3]
        top3 = [(le.inverse_transform([i])[0], probabilities[i]*100) for i in top3_idx]
    else:
        confidence = 100.0
        top3 = [(disease_name, 100.0)]

    # Severity & risk level
    if sev_score <= 13:   risk_level = 'Low'
    elif sev_score <= 26: risk_level = 'Medium'
    else:                 risk_level = 'High'

    urgency_msg   = URGENCY_MSG[risk_level]
    category, dept= DEPARTMENT_MAP.get(disease_name, ('General', 'General Medicine'))
    description   = desc_dict.get(disease_name, 'Description not available.')
    precautions   = prec_dict.get(disease_name, [])

    # ── Print Report ───────────────────────────────────────────
    print(f"\n{'─'*60}")
    print(f"  ALL MODEL PREDICTIONS (Consensus View)")
    print(f"{'─'*60}")
    print(f"  {'Model':<25} {'Prediction'}")
    print(f"  {'-'*45}")
    print(f"  {'Decision Tree':<25} {pred_dt}")
    print(f"  {'Random Forest':<25} {pred_rf}")
    print(f"  {'Logistic Regression':<25} {pred_lr}")
    print(f"  {'SVM (RBF)':<25} {pred_svm}")
    print(f"  {'ANN (Keras)':<25} {pred_ann}")
    print(f"  {'CNN (LeNet)':<25} {pred_cnn}")

    print(f"\n{'─'*60}")
    print(f"  PRIMARY PREDICTION (via {best_model_name})")
    print(f"{'─'*60}")
    print(f"  Disease    : {disease_name}")
    print(f"  Category   : {category}")
    print(f"  Confidence : {confidence:.1f}%")

    print(f"\n{'─'*60}")
    print(f"  RISK ASSESSMENT")
    print(f"{'─'*60}")
    print(f"  Severity Score : {sev_score}")
    print(f"  Risk Level     : {urgency_msg}")

    print(f"\n{'─'*60}")
    print(f"  RECOMMENDED ACTION")
    print(f"{'─'*60}")
    print(f"  See a          : {dept} Specialist")
    print(f"\n  About this condition:")
    print(f"  {description}")
    if precautions:
        print(f"\n  Precautionary advice:")
        for i, p in enumerate(precautions, 1):
            print(f"    {i}. {str(p).strip().title()}")

    if top3:
        print(f"\n{'─'*60}")
        print(f"  TOP 3 POSSIBILITIES")
        print(f"{'─'*60}")
        for d, prob in top3:
            bar = '█' * int(prob / 5)
            print(f"  {d:<35} {prob:5.1f}% {bar}")

    print("═"*60)

    return {
        'disease': disease_name, 'confidence': confidence,
        'risk_level': risk_level, 'department': dept,
        'severity_score': sev_score, 'top3': top3
    }

print(" carebridge_predict() function ready!")

In [ ]:
# ── Test Case 1: Diabetes symptoms ────────────────────────────
result1 = carebridge_predict(
    symptom_list=['polyuria', 'fatigue', 'weight_loss',
                  'blurred_and_distorted_vision', 'irregular_sugar_level', 'excessive_hunger'],
    patient_name="Rahul Sharma (25M)"
)

In [ ]:
# ── Test Case 2: Hepatitis symptoms ───────────────────────────
result2 = carebridge_predict(
    symptom_list=['yellowish_skin', 'dark_urine', 'loss_of_appetite',
                  'abdominal_pain', 'fatigue', 'nausea'],
    patient_name="Priya Verma (32F)"
)

In [ ]:
# ── Test Case 3: Respiratory symptoms ─────────────────────────
result3 = carebridge_predict(
    symptom_list=['breathlessness', 'cough', 'mucoid_sputum',
                  'chest_pain', 'fatigue'],
    patient_name="Arjun Singh (30M)"
)

---
## Phase 6 — Statistical Analysis (FDA)

In [ ]:
# Descriptive Statistics
desc_stats = df.groupby('Disease')['severity_score'].agg(
    Mean='mean', Std_Dev='std', Min='min', Max='max', Count='count').round(2)
desc_stats['Risk_Category'] = desc_stats['Mean'].apply(
    lambda x: 'Low' if x <= 13 else 'Medium' if x <= 26 else 'High')
print("Descriptive Statistics — Severity Score by Disease")
print("="*65)
print(desc_stats.sort_values('Mean', ascending=False).to_string())

fig, ax = plt.subplots(figsize=(12, 12))
sorted_stats = desc_stats.sort_values('Mean', ascending=True).tail(20)
colors_bar = ['#d62728' if v == 'High' else '#ff7f0e' if v == 'Medium' else '#2ca02c'
              for v in sorted_stats['Risk_Category']]
bars = ax.barh(sorted_stats.index, sorted_stats['Mean'],
               color=colors_bar, edgecolor='white')
ax.errorbar(sorted_stats['Mean'], range(len(sorted_stats)),
            xerr=sorted_stats['Std_Dev'], fmt='none',
            color='black', capsize=3, linewidth=1)
ax.set_xlabel("Mean Severity Score (± 1 SD)", fontsize=11)
ax.set_title("Top 20 Diseases by Mean Severity Score", fontsize=14, fontweight='bold')
ax.axvline(x=13, linestyle='--', alpha=0.5)
ax.axvline(x=26, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('plot7_clean_severity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ANOVA Test
severity_groups = [df[df['Disease'] == d]['severity_score'].values for d in OUR_DISEASES]
f_stat, p_value_anova = stats.f_oneway(*severity_groups)
print("One-Way ANOVA — Severity Score Across All Diseases")
print("="*55)
print(f"F-statistic : {f_stat:.4f}")
print(f"p-value     : {p_value_anova:.10f}")
if p_value_anova < 0.05:
    print("\n   Result: p < 0.05 → REJECT H0")
    print("   At least one disease has a significantly different severity score.")

fig, ax = plt.subplots(figsize=(14, 7))
bp = ax.boxplot(severity_groups,
                labels=[format_disease_name(d) for d in OUR_DISEASES],
                patch_artist=True,
                medianprops=dict(color='red', linewidth=2))
colors_box = sns.color_palette("Set3", len(OUR_DISEASES))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
ax.set_ylabel("Severity Score", fontsize=12)
ax.set_title(f"ANOVA: Severity Score Distribution (F={f_stat:.2f}, p={p_value_anova:.2e})",
             fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('plot8_anova_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chi-Square Test
contingency_table = pd.crosstab(df['Disease'], df['risk_level'])
chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency_table)
print(f"Chi-Square Test of Independence")
print(f"Chi² statistic  : {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value          : {p_chi2:.10f}")
if p_chi2 < 0.05:
    print("\n   Result: p < 0.05 → Disease and Risk Level are SIGNIFICANTLY ASSOCIATED.")

fig, ax = plt.subplots(figsize=(12, 6))
contingency_pct = contingency_table.div(contingency_table.sum(axis=1), axis=0) * 100
contingency_pct.plot(kind='bar', stacked=True, ax=ax,
                     color=['#2ecc71', '#f39c12', '#e74c3c'],
                     edgecolor='white', width=0.7)
ax.set_title(f"Chi-Square: Disease vs Risk Level (χ²={chi2:.2f}, p={p_chi2:.2e})",
             fontsize=13, fontweight='bold')
ax.set_ylabel("Percentage of Patients (%)")
ax.set_xticklabels([format_disease_name(d) for d in contingency_table.index],
                   rotation=45, ha='right')
ax.legend(title='Risk Level', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig('plot9_chisquare.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Pearson Correlation Analysis
binary_with_score = X.copy()
binary_with_score['severity_score'] = df['severity_score'].values
correlations = binary_with_score.drop(columns=['severity_score']).corrwith(
    binary_with_score['severity_score']).sort_values(ascending=False)

top_corr    = correlations.head(15)
bottom_corr = correlations.tail(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
colors_pos = ['#d62728' if v > 0.3 else '#ff7f0e' if v > 0.1 else '#aec7e8' for v in top_corr.values]
axes[0].barh(top_corr.index[::-1], top_corr.values[::-1], color=colors_pos[::-1])
axes[0].set_title("Symptoms Positively Correlated with High Severity", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Pearson Correlation Coefficient")
for i, v in enumerate(top_corr.values[::-1]):
    axes[0].text(v + 0.003, i, f'{v:.3f}', va='center', fontsize=9)

axes[1].barh(bottom_corr.index[::-1], bottom_corr.values[::-1], color='#aec7e8')
axes[1].set_title("Symptoms with Lowest Correlation with Severity", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Pearson Correlation Coefficient")
for i, v in enumerate(bottom_corr.values[::-1]):
    axes[1].text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)

plt.suptitle("Pearson Correlation: Symptoms vs Severity Score", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot10_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 7 — Final Summary

In [ ]:
print("╔══════════════════════════════════════════════════════════════╗")
print("║          CareBridge — AI/ML Project Results Summary         ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║  CLASSIFICATION MODELS                                      ║")
print(f"║  Decision Tree Accuracy     : {dt_acc*100:6.2f}%                      ║")
print(f"║  Random Forest Accuracy     : {rf_acc*100:6.2f}%                      ║")
print(f"║  Logistic Regression Acc.   : {lr_acc*100:6.2f}%                      ║")
print(f"║  SVM (RBF) Accuracy         : {svm_acc*100:6.2f}%                      ║")
print(f"║  ANN (Keras/TF) Accuracy    : {ann_acc*100:6.2f}%                      ║")
print(f"║  CNN (LeNet-inspired) Acc.  : {cnn_acc*100:6.2f}%                      ║")
print(f"║  Best Model                 : {best_model_name:<28}║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║  REGRESSION MODEL                                           ║")
print(f"║  Linear Regression R²       : {r2:.4f} (Severity Pred.)       ║")
print(f"║  Linear Regression RMSE     : {rmse:.4f}                          ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║  UNSUPERVISED LEARNING                                      ║")
print(f"║  K-Means Clusters           : {OPTIMAL_K} disease families              ║")
print(f"║  PCA Variance (2 comps)     : {sum(pca.explained_variance_ratio_)*100:.1f}%                     ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║  FDA STATISTICAL TESTS                                      ║")
print(f"║  ANOVA  p-value : {p_value_anova:.2e}  → Diseases differ (p<0.05)   ║")
print(f"║  Chi²   p-value : {p_chi2:.2e}  → Associated (p<0.05)       ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║  DATASET SUMMARY                                            ║")
print(f"║  Total patient records      : {len(df):<5} rows                       ║")
print(f"║  Features (symptoms)        : {X.shape[1]:<5} binary columns             ║")
print(f"║  Diseases covered           : {NUM_CLASSES:<5}                            ║")
print("║  Risk levels                : Low / Medium / High           ║")
print("╚══════════════════════════════════════════════════════════════╝")

In [ ]:
# ─────────────────────────────────────────────────────────────
# INTERACTIVE DEMO — Try your own symptoms!
# ─────────────────────────────────────────────────────────────
name = input("Enter patient name: ")
symptoms_input = input("Enter symptoms (comma-separated): ")
symptom_list = [s.strip() for s in symptoms_input.split(',')]
carebridge_predict(symptom_list, patient_name=name)